# CLOVA Studio 429(Too Many Requests) 한도 실측

**목적** — HyperCLOVA X 호출에 어느 정도의 지연/백오프를 넣어야 하는지를 추측이 아니라 실측으로 정한다.

측정 순서
1. 한도 헤더 읽기 (`x-ratelimit-*`) — 모델·엔드포인트별 한도
2. 창(window) 동작 확인 — 고정창인지, 리셋이 어떻게 도는지
3. 429 유발 — 응답 헤더에 `Retry-After`가 오는지
4. 백오프 전략 비교 — (A) 지수 백오프 / (B) reset 헤더 기반 대기 / (C) 다른 모델 우회
5. 15초 SLA(AGENTS.md 절대규칙 5) 기준 권고값 산출

> 주의: 3~4단계는 실제로 한도를 소진시킨다. 평가/데모 직전에는 돌리지 말 것.
> `RUN_HEAVY = False`로 두면 1~2단계(호출 3회)만 돈다.

## 측정 결론 (2026-09-03 실측, HCX-007 기준)

| 항목 | 실측값 |
|---|---|
| 요청 한도 | **60 req/min** (HCX-007·HCX-005·bge-m3 각각) / 토큰 60,000 tok/min (임베딩 40,000) |
| 창 방식 | **고정창(fixed window) 60s**. `reset` 카운트다운 후 한 번에 59로 복귀. 슬라이딩·토큰버킷 아님 |
| 카운터 범위 | **모델별 분리** — HCX-007 소진 상태에서 HCX-005는 즉시 200 |
| 429 응답 | body `code: 42901`, **`Retry-After` 없음**, `x-ratelimit-reset-requests`만 제공 |
| A. 지수 백오프 (openai SDK 기본 = `ChatClovaX` 내부) | 8회 시도 · 35.9s · **전부 429 실패** |
| B. `reset` 헤더만큼 대기 | 성공하지만 **60.0s** 소요 |
| C. 예비 모델(HCX-005) 우회 | **즉시 200** — 15초 SLA 안에서 유일하게 유효 |
| 단건 지연 | 0.3~0.4s (버스트 시 최대 5.4s까지 관측) |
| 파이프라인 소비량 | 질의 1건 ≈ 7 LLM 호출 + 임베딩 1회 → **지속 가능 ≈ 8.6 질의/분** |

**결론** — 창이 한 번 마르면 최대 60초 대기가 강제되므로 어떤 백오프 값(0.5s·1s·2s…)도 15초 SLA를 못 지킨다.
"429 후 얼마나 기다릴지"가 아니라 **"429가 나기 전에 늦춘다"**(선제 페이싱 + 예비 모델 우회)가 정답이다.

## 골드셋 기반 Agent 호출 테스트 코드에 반드시 넣어야 하는 원칙

골드셋 35문항 × 7호출 ≈ **245 req** → 60 req/min 한도에서 이론상 최소 4.1분. 아래를 지키지 않으면 테스트 결과에 429가 섞여 정답률이 왜곡된다.

1. **질의는 순차 실행. `ThreadPoolExecutor`/`asyncio.gather`로 문항을 동시 실행하지 않는다.**
   E2E 19s 순차면 ≈ 22 req/min으로 한도의 1/3이다. 동시성 2 이상부터 창 소진 위험이 생긴다.
2. **문항 사이 최소 간격 7초**(`60s ÷ 8.6질의`) **를 보장한다.** 파이프라인이 그보다 오래 걸리면 추가 sleep 0, 빨리 끝나면(캐시·abstain 경로) 남은 만큼만 sleep한다 — `sleep(max(0, 7 - elapsed))`.
3. **응답 헤더의 `x-ratelimit-remaining-requests`를 읽어 `remaining ≤ 3`이면 리셋(`reset`초)까지 쉰다.** 6단계 `ClovaRateGuard`를 그대로 쓴다.
4. **429는 실패로 기록하되 재시도 루프를 돌리지 않는다.** 재시도는 같은 창 안에서 카운터만 더 태우고(A 실측) 다음 문항까지 오염시킨다. `ChatClovaX(max_retries=0)`로 SDK 내부 재시도도 끈다.
5. **429가 난 문항은 `status=rate_limited`로 따로 표기하고 정답률 분모에서 분리한다.** 오답과 섞이면 원인 분석이 불가능하다.
6. **테스트 시작 전 `remaining`을 1회 확인한다.** 직전 실험(이 노트북 3~4단계 등)이 창을 비워 둔 상태면 `reset`초를 기다리고 시작한다.
7. **임베딩 40,000 tok/min을 별도 카운터로 본다.** 문항당 임베딩 1회(수십 토큰)라 골드셋 규모에선 여유가 있지만, 벡터 리빌드 스크립트와 동시에 돌리지 않는다.
8. **테스트 직전에 이 노트북 3~4단계(`RUN_HEAVY=True`)를 돌리지 않는다.** 한도를 의도적으로 소진하는 셀이다.

In [ ]:
# ── 0단계: 공통 셋업 ────────────────────────────────────────────────────────
# langchain_naver(ChatClovaX)는 openai SDK로 감싸져 있어서 429 응답 헤더가
# 호출부까지 올라오지 않는다. 한도 자체를 재려면 requests로 직접 때려야 한다.
import os
import time
import random
import collections
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests
from dotenv import load_dotenv

# 노트북 위치(test/ratelimit-test/)에서 repo 루트를 거슬러 올라가 .env를 읽는다.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists())
load_dotenv(ROOT / ".env")

# .env는 CLOVA_API_KEY, langchain_naver는 CLOVASTUDIO_API_KEY를 본다(둘 다 허용).
API_KEY = os.getenv("CLOVASTUDIO_API_KEY") or os.getenv("CLOVA_API_KEY")
assert API_KEY, ".env에 CLOVA_API_KEY(또는 CLOVASTUDIO_API_KEY)가 없다"

BASE = "https://clovastudio.stream.ntruss.com/v1/openai"
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
CHAT_MODEL = "HCX-007"     # 파이프라인 본선 모델
SPARE_MODEL = "HCX-005"    # 우회 후보

RUN_HEAVY = True           # False면 한도를 소진하는 3~4단계를 건너뛴다


def chat(model=CHAT_MODEL, prompt="1", max_tokens=1, timeout=30):
    """가장 싼 chat 호출 1회. 상태코드·지연·한도 헤더만 뽑아 돌려준다."""
    t0 = time.time()
    r = requests.post(
        f"{BASE}/chat/completions", headers=HEADERS, timeout=timeout,
        json={"model": model, "messages": [{"role": "user", "content": prompt}],
              "max_completion_tokens": max_tokens, "temperature": 0},
    )
    return {
        "status": r.status_code,
        "elapsed": time.time() - t0,
        # 한도 정보는 성공/실패 응답 양쪽에 다 실려 온다.
        "limit": r.headers.get("x-ratelimit-limit-requests"),
        "remaining": r.headers.get("x-ratelimit-remaining-requests"),
        "reset": r.headers.get("x-ratelimit-reset-requests"),
        "retry_after": r.headers.get("Retry-After"),   # 있으면 SDK가 이걸 우선 존중한다
        "headers": dict(r.headers),
        "body": r.text[:300],
    }


print("repo root :", ROOT)
print("key loaded:", bool(API_KEY), "| RUN_HEAVY =", RUN_HEAVY)

## 1단계 — 한도 헤더 읽기

한도는 문서를 뒤질 필요 없이 **모든 응답 헤더에 실려 온다**. 세 갈래(본선 모델 / 예비 모델 / 임베딩)를 각각 1회씩 찔러 본다.

In [ ]:
# ── 1단계: 엔드포인트·모델별 한도 확인 ─────────────────────────────────────
def limits_of(kind, **kw):
    """chat/embeddings 각각 1회 호출해 x-ratelimit-* 를 걷어 온다."""
    if kind == "embeddings":
        r = requests.post(f"{BASE}/embeddings", headers=HEADERS, timeout=30,
                          json={"model": kw.get("model", "bge-m3"), "input": "한도 확인"})
        h = r.headers
        return dict(status=r.status_code, req=h.get("x-ratelimit-limit-requests"),
                    tok=h.get("x-ratelimit-limit-tokens"), reset=h.get("x-ratelimit-reset-requests"))
    res = chat(**kw)
    return dict(status=res["status"], req=res["limit"],
                tok=res["headers"].get("x-ratelimit-limit-tokens"), reset=res["reset"])


rows = [
    (f"chat {CHAT_MODEL}", limits_of("chat", model=CHAT_MODEL)),
    (f"chat {SPARE_MODEL}", limits_of("chat", model=SPARE_MODEL)),
    ("embeddings bge-m3", limits_of("embeddings")),
]
print(f"{'대상':22s} {'status':>6s} {'req/min':>8s} {'tok/min':>9s} {'reset':>6s}")
for name, v in rows:
    print(f"{name:22s} {v['status']:>6} {str(v['req']):>8} {str(v['tok']):>9} {str(v['reset']):>6}")

## 2단계 — 창(window) 동작 확인

한도가 **고정창(fixed window)** 인지 **슬라이딩/토큰버킷**인지에 따라 대응이 완전히 달라진다.
- 고정창이면 남은 수가 0이 되는 순간부터 리셋까지는 무슨 짓을 해도 429다 → 백오프로 못 넘긴다.
- 토큰버킷이면 조금만 기다려도 토큰이 돌아온다 → 짧은 백오프로 넘긴다.

연속 호출하며 `remaining` 감소 패턴과 `reset` 초를 같이 본다.

In [ ]:
# ── 2단계: remaining 감소·reset 관측 ───────────────────────────────────────
N = 12   # 한도(60)의 20%만 소모한다
t0 = time.time()
for i in range(1, N + 1):
    r = chat()
    print(f"{i:3d} t={time.time()-t0:5.1f}s status={r['status']} "
          f"remaining={r['remaining']:>3} reset={r['reset']} lat={r['elapsed']:.2f}s")

# 리셋이 '초 단위 카운트다운'이면 고정창, 호출마다 60s로 되돌아오면 슬라이딩이다.
print("\n해석: reset 값이 호출마다 줄어들면 창 시작 시각이 고정된 fixed window,"
      "\n      매번 60s면 요청마다 다시 세는 sliding window다.")

## 3단계 — 429 유발과 응답 형태

동시 요청을 창 한도 이상으로 던져 실제 429를 받아 본다. 확인 포인트는 하나다: **`Retry-After` 헤더가 오는가?**
openai SDK(ChatClovaX의 내부)는 `Retry-After`가 있으면 그 값을 그대로 존중하고, 없으면 `0.5 * 2^n`(최대 8s) 지수 백오프로 떨어진다.

In [ ]:
# ── 3단계: 동시 버스트로 429 유발 ──────────────────────────────────────────
def burst(n=80, model=CHAT_MODEL):
    """n개를 동시에 던지고 상태코드 분포를 센다."""
    with ThreadPoolExecutor(max_workers=n) as ex:
        return list(ex.map(lambda _: chat(model=model), range(n)))


if RUN_HEAVY:
    t0 = time.time()
    res = burst(80)
    counts = collections.Counter(r["status"] for r in res)
    lat200 = sorted(r["elapsed"] for r in res if r["status"] == 200)
    print(f"동시 80건 / wall {time.time()-t0:.1f}s / 상태 분포: {dict(counts)}")
    if lat200:
        print(f"200 지연 min/med/max = {lat200[0]:.2f}/{lat200[len(lat200)//2]:.2f}/{lat200[-1]:.2f}s"
              "  ← 단건 0.35s 대비 버스트 시 지연(실측 최대 5.4s까지 튄 적 있음)")

    sample = next((r for r in res if r["status"] == 429), None)
    if sample:
        print("\n[429 응답]")
        print("  body        :", sample["body"])
        print("  Retry-After :", sample["retry_after"], " ← None이면 SDK는 지수 백오프로 떨어진다")
        print("  remaining   :", sample["remaining"])
        print("  reset       :", sample["reset"], " ← 실제로 기다려야 하는 시간은 이 값이다")
else:
    print("RUN_HEAVY=False → 건너뜀")

## 4단계 — 백오프 전략 3종 비교

한도를 소진시킨 직후, 세 방식이 각각 **몇 초 만에 200을 받아내는지** 잰다.

| | 전략 | 구현 위치 |
|---|---|---|
| A | 지수 백오프 `0.5·2^n`, jitter, 최대 8s (openai SDK 기본값) | `ChatClovaX(max_retries=N)` |
| B | `x-ratelimit-reset-requests` 만큼 대기 | 직접 구현해야 함 |
| C | 429면 다른 모델로 우회 | 직접 구현해야 함 |

In [ ]:
# ── 4단계: 백오프 전략 비교 ────────────────────────────────────────────────
def saturate(n=70):
    """창을 비운다. 이후 호출은 리셋 전까지 전부 429."""
    burst(n)


results = {}
if RUN_HEAVY:
    # A. openai SDK와 동일한 지수 백오프 (0.5, 1, 2, 4, 8, 8 ... + jitter)
    saturate()
    t0 = time.time()
    for n in range(8):
        r = chat()
        if r["status"] == 200:
            results["A"] = (n + 1, time.time() - t0, True)
            break
        delay = min(0.5 * 2 ** n, 8.0) * (1 - 0.25 * random.random())
        print(f"  A 시도{n+1}: 429 (reset={r['reset']}) → {delay:.2f}s 대기")
        time.sleep(delay)
    else:
        results["A"] = (8, time.time() - t0, False)
    print(f"A 지수백오프: 시도 {results['A'][0]}회 / {results['A'][1]:.1f}s / 성공={results['A'][2]}\n")

    time.sleep(65)  # 다음 실험을 위해 창을 완전히 비운다

    # B. reset 헤더만큼 기다린다 (정답이지만 느리다)
    saturate()
    t0 = time.time()
    for n in range(3):
        r = chat()
        if r["status"] == 200:
            results["B"] = (n + 1, time.time() - t0, True)
            break
        wait = float((r["reset"] or "60s").rstrip("s")) + 0.5
        print(f"  B 시도{n+1}: 429 (reset={r['reset']}) → {wait:.1f}s 대기")
        time.sleep(wait)
    print(f"B reset기반: 시도 {results['B'][0]}회 / {results['B'][1]:.1f}s / 성공={results['B'][2]}\n")

    # C. 429면 다른 모델로 우회 (모델별로 카운터가 따로인지 확인)
    saturate()
    a, b = chat(CHAT_MODEL), chat(SPARE_MODEL)
    results["C"] = (a["status"], b["status"])
    print(f"C 모델우회: {CHAT_MODEL}={a['status']}(rem={a['remaining']}) "
          f"{SPARE_MODEL}={b['status']}(rem={b['remaining']})")
    print("  → 상태가 갈리면 카운터는 모델별로 분리돼 있고, 우회가 유효한 탈출구다")
else:
    print("RUN_HEAVY=False → 건너뜀")

## 5단계 — 15초 SLA 기준 권고값 산출

AGENTS.md 절대규칙 5: **응답 15초 이내**. 백오프에 쓸 수 있는 시간은 "15초 − 파이프라인 실제 소요"뿐이다.
질의 1건이 소비하는 호출 수와 한도를 놓고 지속 가능한 처리량을 계산한다.

In [ ]:
# ── 5단계: 권고값 계산 ─────────────────────────────────────────────────────
RPM = 60          # 1단계 실측값
CALLS_PER_Q = 7   # intent·verify·concept·SQL 2회·graph·answer 등 (src/agent/nodes.py 기준 대략치)
E2E_SEC = 19      # 현재 E2E 실측(18~20s)

qpm = RPM / CALLS_PER_Q
print(f"지속 가능 처리량: {RPM} req/min ÷ {CALLS_PER_Q} call/질의 = 질의 {qpm:.1f}건/분 "
      f"(= 질의 간 최소 {60/qpm:.1f}s)")
print(f"호출 간 균등 페이싱을 걸면 최소 간격 {60/RPM:.2f}s → 질의 1건에 페이싱만 "
      f"{CALLS_PER_Q * 60/RPM:.1f}s가 붙는다")
print(f"15초 예산 대비 여유: {15 - E2E_SEC:+.0f}s "
      "→ 이미 초과 상태이므로 '호출마다 sleep'은 쓸 수 없다\n")

budget = max(0.0, 15 - E2E_SEC)
print(f"429 재시도에 허용되는 총 대기: {budget:.1f}s")
print("실측상 창이 마른 뒤 회복까지는 최대 60s가 필요하다 → 백오프로는 절대 못 덮는다.")
print("결론: '429가 난 뒤 얼마나 기다릴까'가 아니라 '429가 나기 전에 늦춘다'가 정답이다.")

## 6단계 — 드롭인 코드 (선제 페이싱 + 제한적 재시도)

핵심은 두 가지다.
1. **응답 헤더의 `remaining`을 계속 본다.** 여유가 있으면 지연 0(15초 SLA를 깎지 않는다), 바닥나기 직전에만 창 리셋까지 잔다.
2. **429가 이미 났으면** 남은 예산 안에서만 재시도하고, 넘어가면 예비 모델로 우회하거나 즉시 실패시킨다.

`ChatClovaX(max_retries=...)`만으로는 부족하다 — SDK의 지수 백오프는 고정창을 넘지 못한다(4단계 A 참조).

In [ ]:
# ── 6단계: 선제 페이싱 가드 (네트워크 없이 검증 가능) ──────────────────────
import threading


class ClovaRateGuard:
    """CLOVA 한도 헤더를 추적해 창이 마르기 전에 스스로 늦추는 가드.

    remaining이 floor보다 남아 있으면 대기 0 - 정상 경로의 지연을 늘리지 않는다.
    바닥에 닿았을 때만 창 리셋까지 자되, budget을 넘는 대기는 -1로 돌려
    호출부가 예비 모델 우회/즉시 실패를 고르게 한다.
    """

    def __init__(self, floor=3, budget=3.0):
        self.floor = floor          # 이만큼은 다른 스레드 몫으로 남긴다
        self.budget = budget        # 15초 SLA에서 대기에 쓸 수 있는 최대 시간
        self._lock = threading.Lock()
        self._remaining = None
        self._reset_at = 0.0

    def observe(self, headers, now=None):
        """응답 헤더에서 잔여량과 창 리셋 시각을 갱신한다."""
        now = time.time() if now is None else now
        rem, reset = headers.get("x-ratelimit-remaining-requests"), headers.get("x-ratelimit-reset-requests")
        if rem is None:
            return
        with self._lock:
            self._remaining = int(rem)
            if reset:
                self._reset_at = now + float(str(reset).rstrip("s"))

    def wait_seconds(self, now=None):
        """호출 직전에 자야 할 시간(초). budget을 넘으면 -1(=우회하라)."""
        now = time.time() if now is None else now
        with self._lock:
            if self._remaining is None or self._remaining > self.floor:
                return 0.0
            wait = max(0.0, self._reset_at - now) + 0.2
        return wait if wait <= self.budget else -1.0


def _demo():
    """헤더 시나리오만으로 가드 동작을 확인한다(호출 없음)."""
    g = ClovaRateGuard(floor=3, budget=3.0)
    assert g.wait_seconds(now=100) == 0.0                      # 관측 전에는 막지 않는다
    g.observe({"x-ratelimit-remaining-requests": "40",
               "x-ratelimit-reset-requests": "50s"}, now=100)
    assert g.wait_seconds(now=100) == 0.0                      # 여유 있으면 지연 0
    g.observe({"x-ratelimit-remaining-requests": "2",
               "x-ratelimit-reset-requests": "2s"}, now=100)
    assert 2.0 < g.wait_seconds(now=100) <= 3.0                # 바닥 + 리셋 임박 → 잠깐 잔다
    g.observe({"x-ratelimit-remaining-requests": "0",
               "x-ratelimit-reset-requests": "55s"}, now=100)
    assert g.wait_seconds(now=100) == -1.0                     # 예산 초과 → 우회 신호
    print("ClovaRateGuard 자체 점검 통과")


_demo()

In [ ]:
# ── 6단계(계속): 실제 호출 래퍼 ────────────────────────────────────────────
GUARD = ClovaRateGuard(floor=3, budget=3.0)


def chat_guarded(model=CHAT_MODEL, spare=SPARE_MODEL, **kw):
    """선제 페이싱 → 호출 → 429면 예산 안에서 1회 재시도 → 그래도 429면 예비 모델."""
    wait = GUARD.wait_seconds()
    if wait > 0:
        time.sleep(wait)                       # 창 리셋이 코앞이면 잠깐 잔다
    elif wait < 0:
        model = spare                          # 60s를 기다릴 순 없다 → 다른 버킷으로

    r = chat(model=model, **kw)
    GUARD.observe(r["headers"])
    if r["status"] != 429:
        return r

    # 여기까지 왔다면 이미 창이 말랐다. 예산 안에서만 한 번 더 본다.
    wait = GUARD.wait_seconds()
    if 0 <= wait <= GUARD.budget:
        time.sleep(max(wait, 0.5))
        r = chat(model=model, **kw)
        GUARD.observe(r["headers"])
        if r["status"] != 429:
            return r
    r = chat(model=spare, **kw)                # 마지막 탈출구: 예비 모델
    GUARD.observe(r["headers"])
    return r


probe = chat_guarded()
print("chat_guarded ->", probe["status"], "remaining:", probe["remaining"])

## 결과 요약

측정값과 결론은 아래 `docs` 또는 커밋 메시지에 함께 남긴다. 재실행 시 `RUN_HEAVY = False`로 두면
1~2단계(호출 3회)만 돌아 한도를 건드리지 않는다.